# GVSAM gallery

Every test image three times: raw, annotation, prediction. Boxes are coloured by
what the evaluation counted them as, using the same matching code, so a red box
here is a false positive in `eval_cellfinder.tsv`.

Stage 7 of `run_pipeline.sh` executes this notebook and writes the result to
`gallery.html`. Opened by hand in Jupyter it works the same way; the paths come
from the environment and fall back to the values below.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from matplotlib.patches import Rectangle

from lib_eval import load_model, match, predict_boxes
from lib_io import inside_border, read_image_any

# Set by run_pipeline.sh. Edit the fallbacks to open the notebook by hand.
CONFIG = Path(os.environ.get("GVSAM_CONFIG", "config.yaml"))
cfg = yaml.safe_load(CONFIG.read_text())
ROOT = Path(os.path.expanduser(cfg["project"]["root"]))

RUN_DIR = Path(os.environ.get("GVSAM_RUN_DIR", ROOT / "04_runs" / cfg["project"]["run"]))
SPLIT_DIR = Path(os.environ.get("GVSAM_SPLIT_DIR", ROOT / "03_splits"))
CKPT = os.path.expanduser(os.environ.get("GVSAM_CKPT", cfg["models"]["cellsam_ckpt"]))

ev = cfg["eval"]
N_IMAGES = int(cfg["visualize"]["n_images"])

# blue annotation, green hit, red false positive, orange missed
COLORS = {"gt": "#5A949C", "tp": "#88C88A", "fp": "#D55958", "fn": "#D76C2D"}

print(f"run {RUN_DIR.name}, score_thr {ev['score_thr']}, matching by {ev['match']}")

## Which images

If stage 4 has already run, the images with the most missed clusters come first.
Those are the ones worth looking at: a good average hides exactly the few images
where the model fails.

In [ ]:
test = pd.read_csv(SPLIT_DIR / "test_bboxes.tsv", sep="\t")
order = sorted(test["uid"].unique())

per_image = RUN_DIR / "eval_per_image.tsv"
if per_image.exists():
    worst = (pd.read_csv(per_image, sep="\t")
             .query("model == 'gvsam'")
             .sort_values(["FN", "FP"], ascending=False)["uid"].tolist())
    order = [u for u in worst if u in set(order)]

uids = order[:N_IMAGES]
print(f"{len(order)} test images, showing {len(uids)}")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = load_model(CKPT, device, str(RUN_DIR / "cellfinder.pt"))
print("model loaded on", device)

## Gallery

Left the raw image, middle the annotation, right the prediction. Green boxes
matched an annotation, red ones did not, and orange marks an annotated cluster
that no box reached.

In [ ]:
def draw(ax, img, title):
    ax.imshow(img, cmap="gray",
              vmin=np.percentile(img, 1), vmax=np.percentile(img, 99))
    ax.set_title(title, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])


def boxes(ax, arr, color, lw=1.2):
    for x1, y1, x2, y2 in arr:
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1,
                               fill=False, edgecolor=color, linewidth=lw))


for uid in uids:
    g = test[test["uid"] == uid]
    img = read_image_any(g["image_path"].iloc[0])
    gt = g[["xmin", "ymin", "xmax", "ymax"]].dropna().to_numpy(np.float32)

    pred, scores = predict_boxes(model, img, ev["score_thr"])
    keep = inside_border(pred, *img.shape, margin=ev["border_margin"])
    pred, scores = pred[keep], scores[keep]

    tp, fp, fn, _, pred_is_tp, gt_is_found = match(
        pred, gt, scores, mode=ev["match"], iou_thr=ev["iou_thr"],
        center_tol=ev["center_tol"], area_ratio=ev["area_ratio"])

    fig, axes = plt.subplots(1, 3, figsize=(13, 4.6))
    draw(axes[0], img, uid)
    draw(axes[1], img, f"annotation, {len(gt)} clusters")
    boxes(axes[1], gt, COLORS["gt"])
    draw(axes[2], img, f"prediction, TP {tp}  FP {fp}  missed {fn}")
    boxes(axes[2], pred[pred_is_tp], COLORS["tp"])
    boxes(axes[2], pred[~pred_is_tp], COLORS["fp"])
    boxes(axes[2], gt[~gt_is_found], COLORS["fn"], lw=1.6)

    plt.tight_layout()
    plt.show()

## What to do with it

- missed clusters, orange, concentrated in dim images: an annotation or contrast
  problem. Lowering `score_thr` buys them at the price of false positives.
- red boxes on cell poles, debris or the medium: raise `score_thr`, then rerun
  stage 4. That takes minutes and needs no retraining.
- red boxes sitting right next to a green one on the same cluster: the detector
  splits one cluster into two. That is a question of what counts as one instance
  in the annotation, and it is fixed in the masks.
- an image with many annotations and no boxes at all: check that the mask really
  belongs to that image. A mismatched pair poisons training and evaluation.